# Préparation des Données — Détection de Fraude Bancaire
**Objectif :** Préparer les données pour l'entraînement du modèle ML.

In [2]:
#  Importation des bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Gestion du déséquilibre
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

sns.set_theme(style="whitegrid")
print("✅ Bibliothèques importées avec succès !")

✅ Bibliothèques importées avec succès !


In [3]:
# Chargement du dataset
df = pd.read_csv(r'C:\Users\emman\Documents\Portfolio\fraud-detection-ml\data\creditcard.csv')
print(f"Dataset chargé : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")

Dataset chargé : 284,807 lignes x 31 colonnes


In [4]:
# Feature Engineering - Création de nouvelles variables

# 1. Heure de la transaction (on avait vu que 2h = pic de fraude)
df['Hour'] = (df['Time'] / 3600) % 24

# 2. Catégorie d'heure (nuit, matin, après-midi, soir)
def categoriser_heure(h):
    if 0 <= h < 6:
        return 0 # Nuit
    elif 6 <= h < 12:
        return 1 # Matin
    elif 12 <= h < 18:
        return 2  # Après-midi
    else:
        return 3  # Soir

df['Hour_Category'] = df['Hour'].apply(categoriser_heure)

# 3. Montant arrondi au log (réduit l'effet des valeurs extrêmes)
df['Amount_Log'] = np.log1p(df['Amount'])

# Vérification
print(" Nouvelles variables créées !")
print(f"\n Aperçu des nouvelles colonnes :")
print(df[['Time', 'Hour', 'Hour_Category', 'Amount', 'Amount_Log']].head(10))

 Nouvelles variables créées !

 Aperçu des nouvelles colonnes :
   Time      Hour  Hour_Category  Amount  Amount_Log
0   0.0  0.000000              0  149.62    5.014760
1   0.0  0.000000              0    2.69    1.305626
2   1.0  0.000278              0  378.66    5.939276
3   1.0  0.000278              0  123.50    4.824306
4   2.0  0.000556              0   69.99    4.262539
5   2.0  0.000556              0    3.67    1.541159
6   4.0  0.001111              0    4.99    1.790091
7   7.0  0.001944              0   40.80    3.732896
8   7.0  0.001944              0   93.20    4.545420
9   9.0  0.002500              0    3.68    1.543298


In [5]:
# 🎯 Séparation des features et de la cible

# X = toutes les colonnes sauf 'Class' (ce qu'on donne au modèle)
X = df.drop(['Class', 'Time'], axis=1)

# y = uniquement la colonne 'Class' (ce qu'on veut prédire)
y = df['Class']

print("✅ Séparation effectuée !")
print(f"\n📊 X (features)  : {X.shape[0]:,} lignes x {X.shape[1]} colonnes")
print(f"🎯 y (cible)     : {y.shape[0]:,} valeurs")
print(f"\n📋 Colonnes utilisées :")
print(list(X.columns))

✅ Séparation effectuée !

📊 X (features)  : 284,807 lignes x 32 colonnes
🎯 y (cible)     : 284,807 valeurs

📋 Colonnes utilisées :
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Hour', 'Hour_Category', 'Amount_Log']


In [6]:
# ✂️ Division Train / Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% pour le test
    random_state=42,    # Pour avoir les mêmes résultats à chaque fois
    stratify=y          # Garder les mêmes proportions de fraudes
)

print("✅ Division effectuée !")
print(f"\n📊 Taille des ensembles :")
print(f"   Train : {X_train.shape[0]:,} lignes ({len(X_train)/len(X)*100:.0f}%)")
print(f"   Test  : {X_test.shape[0]:,} lignes ({len(X_test)/len(X)*100:.0f}%)")
print(f"\n🚨 Fraudes dans le train : {y_train.sum():,}")
print(f"🚨 Fraudes dans le test  : {y_test.sum():,}")

✅ Division effectuée !

📊 Taille des ensembles :
   Train : 227,845 lignes (80%)
   Test  : 56,962 lignes (20%)

🚨 Fraudes dans le train : 394
🚨 Fraudes dans le test  : 98


In [7]:
# 📏 Normalisation avec StandardScaler
scaler = StandardScaler()

# On normalise uniquement sur le train
# puis on applique la même transformation sur le test
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled['Amount'] = scaler.fit_transform(
    X_train[['Amount']]
)
X_test_scaled['Amount'] = scaler.transform(
    X_test[['Amount']]
)

print("✅ Normalisation effectuée !")
print(f"\n📊 Colonne Amount AVANT normalisation :")
print(f"   Moyenne : {X_train['Amount'].mean():.2f}€")
print(f"   Std     : {X_train['Amount'].std():.2f}€")
print(f"\n📊 Colonne Amount APRÈS normalisation :")
print(f"   Moyenne : {X_train_scaled['Amount'].mean():.4f}")
print(f"   Std     : {X_train_scaled['Amount'].std():.4f}")

✅ Normalisation effectuée !

📊 Colonne Amount AVANT normalisation :
   Moyenne : 88.18€
   Std     : 250.72€

📊 Colonne Amount APRÈS normalisation :
   Moyenne : 0.0000
   Std     : 1.0000


In [8]:
# ⚖️ Gestion du déséquilibre avec SMOTETomek
print("⏳ Application de SMOTETomek en cours...")
print("   (Cela peut prendre 1-2 minutes, sois patient !)\n")

smt = SMOTETomek(random_state=42)
X_train_resampled, y_train_resampled = smt.fit_resample(
    X_train_scaled, y_train
)

print("✅ SMOTETomek appliqué avec succès !")
print(f"\n📊 AVANT rééchantillonnage :")
print(f"   Normales : {(y_train == 0).sum():,}")
print(f"   Fraudes  : {(y_train == 1).sum():,}")
print(f"\n📊 APRÈS rééchantillonnage :")
print(f"   Normales : {(y_train_resampled == 0).sum():,}")
print(f"   Fraudes  : {(y_train_resampled == 1).sum():,}")

⏳ Application de SMOTETomek en cours...
   (Cela peut prendre 1-2 minutes, sois patient !)

✅ SMOTETomek appliqué avec succès !

📊 AVANT rééchantillonnage :
   Normales : 227,451
   Fraudes  : 394

📊 APRÈS rééchantillonnage :
   Normales : 227,451
   Fraudes  : 227,451
